# 02 SQLite 建库与写入

## 目的
- 建立本地数据库并完成结构化入库，支持后续 SQL 分析与增量更新。

## 方法
- 创建 `macro_data`、`stock_price`、`stock_info` 等核心表。
- 将缓存数据标准化后写入 SQLite。
- 检查表记录数，并执行宏观增量更新（若配置 FRED key）。

## 结果
- 生成 `fin_data.db`，完成主数据表与日志表初始化。
- 输出可验证的表规模信息。

## 结论
- 本阶段完成“数据落库与可维护更新”，为分析环节建立统一数据底座。

In [1]:
# 本单元：导入数据库构建与增量更新所需依赖。
from pathlib import Path
import sqlite3
import pandas as pd

from topic10_workflow import (
    bootstrap_project_from_cache,
    create_tables,
    incremental_update_macro,
    sqlite_connect,
)

project_dir = Path('.').resolve()
part2_output_dir = project_dir / 'cache' / 'a_share'
db_path = project_dir / 'fin_data.db'

In [2]:
# 本单元：执行初始化建库并写入基础数据。
# 初次建库 + 写入
result = bootstrap_project_from_cache(project_dir, part2_output_dir)
print(result)

{'macro_rows': 1890, 'stock_price_rows': 40845, 'stock_info_rows': 10, 'dq_price_anomaly': 13, 'dq_zero_volume': 485}


In [3]:
# 本单元：检查核心数据表记录数以验证写入结果。
# 检查三张主表记录数
conn = sqlite_connect(db_path)
for table in ['macro_data', 'stock_price', 'stock_info', 'update_log', 'data_quality']:
    cnt = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn).iloc[0,0]
    print(table, cnt)
conn.close()

macro_data 1890
stock_price 40845
stock_info 10
update_log 19
data_quality 498


In [4]:
# 本单元：执行宏观数据增量更新（需配置 FRED_API_KEY）。
# 宏观数据增量更新（需要 FRED_API_KEY）
conn = sqlite_connect(db_path)
try:
    inserted = incremental_update_macro(conn)
    print('macro incremental inserted:', inserted)
except Exception as e:
    print('incremental_update_macro skipped/error:', e)
finally:
    conn.close()

macro incremental inserted: 0


In [5]:
# 本单元：自动生成本 Notebook 的真实结果结论（随运行结果变化）。
from IPython.display import Markdown, display

conn = sqlite_connect(db_path)
_counts = {}
for _t in ['macro_data', 'stock_price', 'stock_info', 'update_log', 'data_quality']:
    _counts[_t] = int(pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {_t}', conn).iloc[0,0])
conn.close()

summary_text = f"""
### 自动结论（真实结果）
- macro_data：{_counts['macro_data']} 行
- stock_price：{_counts['stock_price']} 行
- stock_info：{_counts['stock_info']} 行
- update_log：{_counts['update_log']} 行
- data_quality：{_counts['data_quality']} 行

结论：数据库已成功建表并写入，且具备可追踪更新日志与质量检测结果。
"""
display(Markdown(summary_text))


### 自动结论（真实结果）
- macro_data：1890 行
- stock_price：40845 行
- stock_info：10 行
- update_log：20 行
- data_quality：498 行

结论：数据库已成功建表并写入，且具备可追踪更新日志与质量检测结果。
